# 20 · Convergence with the number of skewers (TNG50-2, the `skewers` sweep)


How many skewers are needed? Five nested lattices of the fiducial dataset (TNG50-2 snapshot 25, along $y$, 1 km/s, the
exact crossings, the integrated profile, the exact frame), $N \times N$ = $32^2$, $64^2$, $128^2$ (the fiducial), $256^2$ and
$512^2$ skewers at $x_i = i\,L/N$: every coarser lattice is a subset of the finer ones, read from **one** dataset of the
campaign's store (the $512^2$ lattice extended the fiducial's $128^2$; `run_production.py --sweep skewers`). The reference is
$512^2$. The difference of a coarser lattice is its **sample variance** and its aliasing: the same skewers, fewer of them,
and a lower transverse Nyquist frequency $\pi N/L$ for $P(k, \mu)$. The jackknife squares are the same regions of the
box for every lattice, so the error of the ratio counts the variance the lattices share.

**Memory.** The $512^2$ lattice holds $2 \times 262{,}144 \times 3{,}949$ pixels (8.3 GB of $\tau$); its 3D bins and transforms take
about 40 GB at peak and the run about 20 minutes (Part B gives the time of each step).


Every run is a folder of `scripts/production/run_production.py` (`<OUT>/<SWEEP>/<run>/run.json`) that names its
lattice in the campaign's skewer store; `load_run` reads it. The statistics follow notebooks 11 and 18 on **one line of
sight** ($y$): the transmission $T = e^{-\tau}$ of every line and of the total, its mean $\bar T$, the overflux
$\delta = T/\bar T - 1$, P1D on its native modes and $P(k, \mu)$ of the total in four wedges
(`summary_stats.run_statistics`). One line of sight gives no ensemble, so the errors come from a **delete-one-block
jackknife**: the lattice is cut into $4 \times 4$ squares of skewers and each sample leaves one square out (the
3D power of the field with that square set to zero, divided by the fraction kept). Every run is compared with the
reference **bin by bin**, as the relative difference $X_{\rm run}/X_{\rm ref} - 1$, with the jackknife error of that
ratio (`compare_runs`): the runs share the skewer positions and the blocks, so the sample variance they share cancels.
The last part gives, per run, **the largest $k$ up to which every bin differs by less than 0.1%, 0.5% and 1%**
(`k_max_table`; P1D on every native mode, $P(k, \mu)$ on every $|k|$ bin).

Colours: the fiducial run of the campaign in black in every notebook; the other runs from light (coarse) to dark (fine)
on one scale shared by notebooks 18–21.


- **Part A — the stored skewers read back:** every run's lattice, pixels, sampling and dataset, and the checks of the
  pairing (every lattice the stride of the largest, bit for bit).
- **Part B — the statistics of every run** and their comparison with the reference.
- **Part C — the mean transmission.**
- **Part D — P1D** of every line and of the total.
- **Part E — the wedges of $P(k, \mu)$** of the total.
- **Part F — the converged range:** the largest $k$ within 0.1%, 0.5% and 1% of the reference, and what it implies.

In [ ]:
import os
from pathlib import Path

# ---- configuration: the sweep (scripts/production/run_production.py --sweep skewers)
SWEEP = "skewers"
RUNS = ["n_0032", "n_0064", "n_0128", "n_0256", "n_0512"]   # coarse to fine
REFERENCE = "n_0512"                              # the largest lattice: the reference
FIDUCIAL = "n_0128"                               # the fiducial run (black in every figure)
ORDER = RUNS
NSK = {"n_0032": 32, "n_0064": 64, "n_0128": 128, "n_0256": 256, "n_0512": 512}
RUN_LABELS = {r: f"{n}$^2$" for r, n in NSK.items()}
TITLE_STEM = "Number of skewers"
LINTHRESH = 0.01                                  # [%]
OUT = Path(os.environ.get("SKEWERJAX_PRODUCTION", Path.home() / "skewerjax_production"))   # the --out of the campaign
N_SUB = None                                      # None: each run's lattice; 32: a quick check on the nested 32^2 sub-lattice
N_BLOCKS = 4                                      # jackknife: N_BLOCKS x N_BLOCKS squares of skewers
MU_EDGES = (0.0, 0.5, 0.8, 0.95, 1.0)             # the four wedges of P(k, mu) (notebooks 10 and 11's wedges of xi)
TOLS = (0.001, 0.005, 0.01)                       # the convergence levels of Part F
X_OF_RUN, X_LABEL, X_SCALE, X_TICKS = {r: n * n for r, n in NSK.items()}, "skewers $N^2$", "log", None

# ---- imports
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import json, sys, time
import numpy as np
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import use_style, use_backend
# use_backend()                                    # interactive figures in a live notebook
use_style()
from skewerjax.utils.plotting import (add_panel_letters, field_names, run_styles, plot_runs_mean_flux, plot_runs_p1d,
                                      plot_runs_wedges, plot_k_max_summary)
from skewerjax.summary_stats import (LatticeWedges, block_labels, compare_runs, format_k_max_table, k_max_table,
                                     rebin_pixels, run_statistics, set_k_edges)
from skewerjax.utils.reference_data import P1D_DESI_DR1, nearest_reference
from skewerjax.utils.paths import PRODUCTION_DIR, PRODUCTION_SCRIPTS_DIR   # the run folders; the reader of the skewer store
sys.path.insert(0, str(PRODUCTION_SCRIPTS_DIR))
from skewer_store import load_run
from skewerjax.utils.plotting import PALETTE

## Part A — The stored skewers read back

- `OUT`, `SWEEP`, `RUNS`: the run folders `<OUT>/<SWEEP>/<run>`, each one lattice of the same dataset.
- `N_SUB` must stay `None` here (the lattices are the sweep).

The check of the nesting: every lattice read back is, skewer by skewer and bit for bit, the stride $512/N$ of the $512^2$
lattice (the same skewers are stored once and read by every lattice).

In [ ]:
RECS = {r: json.loads((OUT / SWEEP / r / "run.json").read_text()) for r in RUNS}
S = {r: load_run(OUT / SWEEP / r, N_SUB) for r in RUNS}          # tau (n_lines, N_sk, N_pix), positions, columns, v, dv, L, u
REF = S[REFERENCE]
L, U, PERIOD = REF.L, REF.u, REF.period                          # [cMpc], aH [km/s per cMpc], v_per [km/s]
N = REF.n_per_axis
Z, A_SNAP = RECS[REFERENCE]["snapshot"]["redshift"], RECS[REFERENCE]["snapshot"]["a"]
LINE_IDS = REF.line_ids
FIELDS = (*LINE_IDS, "total")
NAMES = field_names(FIELDS)
print(f"{'run':<13} {'sim':<8} {'lattice':>8} {'N_pix':>6} {'dv [km/s]':>10} {'dx [cMpc]':>10} {'dx [km/s]':>10} {'route':<10} "
      f"{'dataset':<11} {'computed':>8} {'wall':>8}")
for r in RUNS:
    rec, s, sm = RECS[r], S[r], RECS[r]["sampling"]
    print(f"{r:<13} {s.sim:<8} {s.n_per_axis:>5}^2 {s.v.size:>6} {s.dv:>10.4f} {sm['points_cMpc']:>10.3g} {sm['points_kms']:>10.3g} "
          f"{sm['route']:<10} {rec['dataset']['hash']:<11} {rec['timings']['computed']:>8} {rec['wall_s']:>7.0f}s")
    assert np.isfinite(s.tau).all(), r
for r in RUNS:                                    # tau < 0: the round-off of the engine (float32), relative to each skewer's peak
    t = S[r].tau
    neg = t < 0
    if neg.any():
        worst = (-t.min(axis=-1) / t.max(axis=-1)).max(axis=-1)       # (n_lines,) the largest |tau_min| / tau_max of a skewer
        print(f"{r}: tau < 0 in {neg.mean():.2%} of the samples (" + ", ".join(
              f"{NAMES[l]}: min {t[j].min():.2g}, |min| / max per skewer <= {worst[j]:.1e}" for j, l in enumerate(LINE_IDS) if t[j].min() < 0) + ")")
print(f"\nreference {REFERENCE}: {REF.sim} snapshot {REF.snap}, z = {Z:.4f}, L = {L:.3f} cMpc, u = {U:.3f} km/s per cMpc (1 km/s = {1 / U:.5f} cMpc), "
      f"v_per = {PERIOD:.2f} km/s; tau finite in every run")
print("sampling of the reference: " + RECS[REFERENCE]["sampling"]["effect"])

assert len({RECS[r]["dataset"]["hash"] for r in RUNS}) == 1, "every lattice must come from one dataset"
key = {tuple(c): i for i, c in enumerate(REF.columns.tolist())}
for r in RUNS:
    st = S[r].n_per_axis
    idx = [key[(a * (N // st), b * (N // st))] for a, b in S[r].columns.tolist()]
    assert np.array_equal(S[r].tau, REF.tau[:, idx]) and np.array_equal(S[r].positions, REF.positions[idx]), r
print(f"every lattice is the stride of the {N}^2 lattice, bit for bit (dataset {RECS[REFERENCE]['dataset']['hash']})")

## Part B — The statistics of every run

The 3D bins are set by the largest lattice (a coarser one is cut at its own Nyquist frequency $\pi N/L$): $|k|$ one bin per mode up to $3.5\,k_f$, steps of $0.4\,k_f$
to 2 cMpc$^{-1}$, then 12 logarithmic bins to the transverse Nyquist frequency $\pi N/L$; the four wedges of
`MU_EDGES` in $\mu = k_\parallel/|k|$. P1D stays on its native modes $k_m = 2\pi m/v_{\rm per}$ for the checks, and is
averaged in 40 logarithmic bins for the figures only. Each line gives the time of each step of a run.

In [ ]:
KF = 2 * np.pi / L
K_NYQ = np.pi * N / L                                            # the transverse Nyquist frequency of the lattice [1/cMpc]
K_EDGES = set_k_edges(KF, k_1=3.5 * KF, k_2=min(2.0, K_NYQ), k_max=K_NYQ, dk_factor=0.4, n_log=12)
K1D_EDGES = np.geomspace(2 * np.pi / PERIOD, np.pi / min(S[r].dv for r in RUNS), 41)   # display bins of P1D [s/km]
P1D_REF = nearest_reference(P1D_DESI_DR1, Z)
K_DESI = float(np.max(P1D_REF["k"]))                             # [s/km]: the largest k of DESI DR1 at the snapshot's z
GRIDS = {}


def grid_for(n, n_pix):
    """The mu wedges of P on an n x n lattice of n_pix pixels (LatticeWedges: the bins of lattice_grid's P_mu alone, the memory of
    512^2 lattices), cached: the same |k| edges and mu wedges for every run."""
    if (n, n_pix) not in GRIDS:
        GRIDS[n, n_pix] = LatticeWedges(n, n_pix, L, k_edges=K_EDGES, mu_edges=MU_EDGES)
    return GRIDS[n, n_pix]


def statistics(s, transmissions=None, n_pix=None, dv=None):
    """run_statistics of one run (or of ``transmissions`` on ``n_pix`` pixels of ``dv``) on its lattice's jackknife blocks
    and bins, T_bar the mean over every pixel (one scale factor: the box), with the time of each step."""
    n_pix = s.v.size if n_pix is None else n_pix
    steps = []
    out = run_statistics(None if transmissions is not None else s.tau, s.dv if dv is None else dv, line_ids=LINE_IDS,
                         labels=block_labels(s.columns, s.n_per_axis, N_BLOCKS), grid=grid_for(s.n_per_axis, n_pix),
                         transmissions=transmissions, a_pix=np.full(n_pix, A_SNAP), log=steps.append)
    return out, steps


STATS, T0 = {}, time.perf_counter()
for r in RUNS:
    t = time.perf_counter()
    STATS[r], steps = statistics(S[r])
    print(f"{r:<13} {time.perf_counter() - t:6.1f} s: " + "; ".join(steps))
CMP = {r: compare_runs(STATS[r], STATS[REFERENCE], k_edges_1d=K1D_EDGES) for r in RUNS if r != REFERENCE}   # display bins
CMP_ALL = {r: compare_runs(STATS[r], STATS[REFERENCE]) for r in RUNS if r != REFERENCE}                  # every mode: the checks
STYLES = run_styles(ORDER, fiducial=FIDUCIAL)
print(f"every run in {time.perf_counter() - T0:.0f} s; P(k, mu) in {len(MU_EDGES) - 1} wedges x {len(K_EDGES) - 1} |k| bins up to "
      f"{K_NYQ:.2f} cMpc^-1; jackknife over {N_BLOCKS} x {N_BLOCKS} squares of {N // N_BLOCKS} x {N // N_BLOCKS} skewers")

## Part C — The mean transmission

$\bar F$ of every line and of the total, per run, as $100\,(\bar F_{\rm run}/\bar F_{\rm ref} - 1)$ [%] with its
jackknife error (the reference is the line at 0; the grey band is $\pm 0.5\%$, the dotted lines $\pm 0.1\%$ and
$\pm 1\%$; the axis is linear within `LINTHRESH`).

In [ ]:
fig, axs = plot_runs_mean_flux(CMP, STYLES, REFERENCE, field_labels=NAMES, labels=RUN_LABELS, linthresh=LINTHRESH,
                               title=f"{TITLE_STEM}: the mean transmission against {RUN_LABELS[REFERENCE]}")
add_panel_letters(axs)
plt.show()
for r, c in CMP_ALL.items():
    print(f"{r:<13} " + "; ".join(f"{NAMES[f]}: {100 * c['mean_flux']['rel'][j]:+.3g} +- {100 * c['mean_flux']['err'][j]:.2g} %"
                                  for j, f in enumerate(FIELDS)))

## Part D — P1D

Above, $k P_{\rm 1D}/\pi$ of the reference (thick, its jackknife error as a band) and of every run; below, the residual
$100\,(P_{\rm run}/P_{\rm ref} - 1)$ [%] with its jackknife error bars, in the same bands as Part C. The grey bands
lie outside $[2\pi/v_{\rm per}, \pi/dv]$ of the reference; the dashed line is the largest $k$ of the DESI DR1 P1D at
the snapshot's redshift. Every lattice has the same pixels: the same modes.

In [ ]:
fig, axs = plot_runs_p1d(CMP, STYLES, REFERENCE, field_labels=NAMES, labels=RUN_LABELS, linthresh=LINTHRESH,
                         k_limits=(2 * np.pi / PERIOD, np.pi / REF.dv),
                         title=f"{TITLE_STEM}: $P_{{\\rm 1D}}$ against {RUN_LABELS[REFERENCE]}")
for a in axs.ravel():
    a.axvline(K_DESI, color=PALETTE["grey"], ls="--", lw=1.5, zorder=0)
add_panel_letters(axs[0])
plt.show()

## Part E — The wedges of $P(k, \mu)$

One column per wedge: above, $P(k, \mu)$ of the total overflux of the reference (its jackknife error as a band) and of
every run at the reference's bin centres; below, the residual in per cent as in Part D. A lattice of $N$ skewers per side stops at $\pi N/L$.

In [ ]:
fig, axs = plot_runs_wedges(CMP, STYLES, REFERENCE, labels=RUN_LABELS, linthresh=LINTHRESH,
                            title=f"{TITLE_STEM}: $P(k, \\mu)$ of the total against {RUN_LABELS[REFERENCE]}")
add_panel_letters(axs[0])
plt.show()

## Part F — The converged range

For every run and statistic, the largest $k$ up to which **every** bin differs from the reference by less than 0.1%,
0.5% and 1% (`k_max_table`: P1D on its native modes, $P(k, \mu)$ on its $|k|$ bins; "$\geq k$" when every bin passes up to
the run's last one; 0 when the first bin already fails). The last columns give the run's last bin and the median
jackknife error of the relative difference: a $k_{\max}$ set by bins whose difference is within that error is limited
by noise, not by the parameter. The figure draws $k_{\max}(0.5\%)$ against the number of skewers.

In [ ]:
ROWS = k_max_table(CMP_ALL, TOLS)
print(f"against {REFERENCE}; P1D on its native modes [s/km], P(k, mu) on its |k| bins [1/cMpc]; '>= k': every bin passes "
      f"up to the run's last one; the DESI DR1 P1D reaches k = {K_DESI:.3f} s/km at z = {P1D_REF['z']}")
print(format_k_max_table(ROWS, TOLS))
fig, axs = plot_k_max_summary(ROWS, X_OF_RUN, xlabel=X_LABEL, xscale=X_SCALE, xticks=X_TICKS, tol=0.005, p1d_k_ref=K_DESI,
                              title=f"{TITLE_STEM}: the largest $k$ converged to 0.5% against {RUN_LABELS[REFERENCE]}")
add_panel_letters(axs)
plt.show()

**Reading.** The sub-lattices are subsets of the $512^2$ skewers. Their differences from it are sample variance,
correlated with the reference, plus aliasing of the transverse modes near each lattice's own Nyquist frequency
$\pi N/L$: the excess of $P(k, \mu)$ at the high-$|k|$ end of every lattice.

* **Mean transmission:** within $2\sigma$ of $512^2$ for every lattice ($\mathrm{Ly}\alpha$ $+0.32 \pm 0.28$% at $32^2$,
  $+0.044 \pm 0.024$% at $256^2$).
* **$P_\mathrm{1D}$ of the total:** within 0.5% up to $k$ = 0.0175 s/km at $128^2$ and 0.13 s/km at $256^2$. $32^2$ and
  $64^2$ already differ at the first modes, by the sample variance of 1,024 and 4,096 skewers.
* **$P(k, \mu)$:** $128^2$ reaches 0.5% only near the line of sight (to 0.24–0.29 cMpc$^{-1}$). $256^2$ reaches
  0.28–0.99 cMpc$^{-1}$. For a statistical quantity 0.5% is a strict test: the residuals are of the size of their
  jackknife errors.

Above 0.7 s/km the $P_\mathrm{1D}$ of $\mathrm{Ly}\alpha$ of $32^2$–$128^2$ lies about 100 times below that of $256^2$ and
$512^2$. That power comes from 0.03% of the skewers (70 of 262,144): sharp edges of saturated absorbers
($b \approx 4$ km/s). Only 2 of them fall in the $128^2$ subset. This is a heavy tail, not a difference in computation:
the median high-$k$ power per skewer is the same ($1.8\times10^{-10}$) for the skewers computed in each run.